### This is the example for this code base

Pls make sure config.json and oai_keys.py are properly configured.

#### 1. load agents from 'multiAgentChat.py'

In [ ]:
import os
import json
import logging
from load_dataset import goat_bench_datasets
from torch.utils.data import DataLoader

from multiAgentChat import Agents, save_yaml

In [ ]:
# load config and prompt
with open('./config.json','r') as f:
    config = json.load(f)

# load prompt from prompt_dict.py
from prompt_dict import prompt

In [ ]:
# define logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
file_handler = logging.FileHandler(config['log_path'])
formatter = logging.Formatter('%(message)s')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# load data
harmfulness_data = goat_bench_datasets(root_path=config['dataset_root_path'])
batch_size = 1
harmfulness_data_loader = DataLoader(harmfulness_data, batch_size=batch_size, shuffle=False, drop_last=False)

# initiate agents, add more agents if needed
agents = Agents(['Internet User','Internet Supervisor'],logger,prompt,config)

#### 2. start chatting within the loop

In [ ]:
from tqdm import tqdm
result_dict = {'id':[], 'gt_label':[], 'pred_label':[], 'text':[], 'error':[]}
for batch in tqdm(harmfulness_data_loader):

    agents.img_path = batch['img_path'][0]
    agents.meme_text = batch['text'][0]
    id = batch['id'][0]
    gt_label = int(batch['label'][0])
    logger.info(f'>>>>>memeID:{id}')
    try:
        agents.initiate_agents()

        agents.chat_for_n_rounds(2)

        # directly summarize and get results from current conversation
        response = agents.summarize_and_get_results()
        
        logger.info('>>>>>endofmemeID')
        logger.info('Summarize and judge:\n'+response)

        pred_label = 1 if 'yes' in response.lower() else 0
        result_dict['id'].append(id)    
        result_dict['gt_label'].append(gt_label)
        result_dict['pred_label'].append(pred_label)

        if gt_label==pred_label:
            logger.info(f'ID:{id} Result: TRUE')
        else:
            logger.info(f'ID:{id} Result: FALSE')
        
    except Exception as e:
        logger.info('Error occurred during conversation...')
        logger.info('Errormessage:',str(e))
        logger.info(f'ID:{id} Result: ERROR')
        result_dict['error'].append(id)
    
    # save dialogues and results
    save_yaml(config, batch, agents, pred_label)

    # reset agents, clear all history
    agents.reset()

#### 3. only run summary after getting the dialogues from step 2

In [ ]:
import yaml
from tqdm import tqdm
import concurrent.futures # for parallel processing
from summary import save_prompt, read_yaml_file_name, get_summary
from prompt_dict import prompt

In [ ]:
#  "harmfulness" "hatefulness" "misogyny" "offensiveness" "sarcasm"
task = 'sarcasm'
prompt = prompt[task]

yaml_root_path = os.path.join('./history', task)
update_save_path = os.path.join('./history_new', task)


# check if the directory exists, if no make it
if not os.path.exists(update_save_path):
    os.makedirs(update_save_path)

# save prompt to yaml as backup
save_prompt(prompt, update_save_path)

# check if the yaml files are already processed
yaml_list_todo = read_yaml_file_name(yaml_root_path)
yaml_list_done = read_yaml_file_name(update_save_path)

def process_yaml_file(yaml_file):
    if yaml_file in yaml_list_done:
        return

    with open(os.path.join(yaml_root_path, yaml_file), 'r') as f:
        yaml_dict = yaml.load(f, Loader=yaml.FullLoader)

    if yaml_dict in yaml_list_done:
        return

    result = get_summary(prompt, yaml_dict)
    yaml_dict['summary']['output'] = result[0]
    yaml_dict['summary']['pred_label'] = 1 if 'yes' in result[0].lower() else 0
    yaml_dict['cost']['completion_tokens'] = result[1]
    yaml_dict['cost']['prompt_tokens'] = result[2]

    with open(os.path.join(update_save_path, yaml_dict['id'] + '.yaml'), 'w') as f:
        yaml.dump(yaml_dict, f, sort_keys=False)

with concurrent.futures.ThreadPoolExecutor(max_workers=30) as executor:
    futures = [executor.submit(process_yaml_file, yaml_file) for yaml_file in yaml_list_todo]

    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
        try:
            future.result()
        except Exception as e:
            print(f"Error processing file: {e}")